# EMBER-Dyn Quickstart 01 — LightGBM

**Dataset**: EMBER-Dyn (T1 binary detection)  
**Target metric**: test ROC AUC ~0.999  
**Runtime**: ~3 minutes on Colab free CPU tier

This notebook reproduces the T1 binary detection baseline from
Table 4 of the EMBER-Dyn paper, using the exact LightGBM
hyper-parameters specified by the EMBER2024 reference
implementation (500 rounds with early stopping, 64 leaves, 100
minimum data points per leaf).

## 1. Install dependencies

In [ ]:
!pip install -q pandas pyarrow scikit-learn lightgbm

## 2. Load EMBER-Dyn static features (Zenodo open tier)

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# EMBER-Dyn open tier is archived on Zenodo (DOI in the README). Download it first
# (e.g. `pip install zenodo_get && zenodo_get <ZENODO_DOI>`) and point DATA_DIR at the
# extracted dataset folder.
DATA_DIR = 'dataset/static'
df_train = pq.read_table(f'{DATA_DIR}/train.parquet').to_pandas()
df_val   = pq.read_table(f'{DATA_DIR}/val.parquet').to_pandas()
df_test  = pq.read_table(f'{DATA_DIR}/test.parquet').to_pandas()
df_chal  = pq.read_table(f'{DATA_DIR}/challenge.parquet').to_pandas()

print(f'Train {len(df_train):,}  Val {len(df_val):,}  Test {len(df_test):,}  Chal {len(df_chal):,}')

## 3. Extract features and labels

In [ ]:
feat_cols = [c for c in df_train.columns if c.startswith('feature_')]
print(f'Feature dimensions: {len(feat_cols)}')  # 2568

def split_X_y(df):
    return df[feat_cols].values.astype('float32'), df['label'].values.astype('int32')

X_train, y_train = split_X_y(df_train)
X_val,   y_val   = split_X_y(df_val)
X_test,  y_test  = split_X_y(df_test)
X_chal,  y_chal  = split_X_y(df_chal)
print(f'X_train shape: {X_train.shape}, label balance: {y_train.mean():.3f}')

## 4. Train LightGBM with the EMBER2024 reference hyperparameters

In [ ]:
import lightgbm as lgb

LGB_PARAMS = dict(
    boosting_type='gbdt',
    num_leaves=64,
    min_data_in_leaf=100,
    learning_rate=0.05,
    n_estimators=500,
    n_jobs=-1,
    random_state=42,
    verbosity=-1,
)
model = lgb.LGBMClassifier(objective='binary', **LGB_PARAMS)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

## 5. Evaluate on the test split (T1 binary detection)

In [ ]:
p_test = model.predict_proba(X_test)[:, 1]
print(f'T1  ROC AUC = {roc_auc_score(y_test, p_test):.4f}    '
      f'PR AUC = {average_precision_score(y_test, p_test):.4f}    '
      f'F1 = {f1_score(y_test, p_test > 0.5):.4f}')

## 6. Challenge-set OOD evaluation (T15 mixed protocol)

Following the EMBER2024 protocol (Section 4.3), we train on
train + val and evaluate on the challenge set joined with the
test-split benign samples (a class-imbalanced 488-sample OOD
evaluation).

In [ ]:
X_tv = np.vstack([X_train, X_val])
y_tv = np.concatenate([y_train, y_val])
model_t15 = lgb.LGBMClassifier(objective='binary', **LGB_PARAMS).fit(X_tv, y_tv)

# Mix challenge (all malicious) with test benigns
test_benign_mask = df_test['label'] == 0
X_eval = np.vstack([X_chal, X_test[test_benign_mask]])
y_eval = np.concatenate([y_chal, df_test.loc[test_benign_mask, 'label'].values])

p_eval = model_t15.predict_proba(X_eval)[:, 1]
print(f'T15 ROC AUC = {roc_auc_score(y_eval, p_eval):.4f}    '
      f'PR AUC = {average_precision_score(y_eval, p_eval):.4f}    '
      f'F1 = {f1_score(y_eval, p_eval > 0.5):.4f}')

## Expected output

```
T1  ROC AUC ~0.999    PR AUC ~0.9998    F1 ~0.992
T15 ROC AUC ~0.86     PR AUC ~0.52      F1 ~0.35
```

Matches paper Table 4 (T1) and Table 5 (T15). The 48-point PR AUC gap
between T1 and T15 is the open evasive-detection challenge described
in §5 of the paper.